In [4]:
# ReAct Prompt Template
REACT_PROMPT_TEMPLATE = """
Please note that you are an intelligent assistant capable of calling external tools.

Available tools are as follows:
{tools}

Please respond strictly in the following format:

Thought: Your thinking process, used to analyze problems, decompose tasks, and plan the next action.
Action: The action you decide to take, must be in one of the following formats:
- {{tool_name}}[{{tool_input}}]`: Call an available tool.
- `Finish[final answer]`: When you believe you have obtained the final answer.
- When you have collected enough information to answer the user's final question, you must use `Finish[final answer]` after the Action: field to output the final answer.

Now, please start solving the following problem:
Question: {question}
History: {history}
"""


In [ ]:
import re
class ReActAgent:
    def __init__(self, llm, tool_executor):
        self.llm = llm
        self.tools = tool_executor

    def run(self,question):
        print("1. run() starts")

        tools = self.tools.toolavaliable()
        print("2. tools:", tools)


        prompt = REACT_PROMPT_TEMPLATE.format(
            tools = tools,
            question= question,
            history= ""
        )
        print("3. prompt:")
        print(prompt)

        message = [{
            "role":"user",
            "content":prompt
        }]
        print("4. message ")

        response =self.llm.think(message)
        print("5. response:", response)

        print(response)
        if not response:
           print("LLM 没有返回有效内容")
           return None

        return response

    def _parse_output(self, text: str):
        part = []
        

        for line in text.split("\n"):
            if line.strip():
               part.append(line)

        think_match = re.search(r"Thought:\s*(.*)",part[-2])
        think = think_match.group(1).strip()

        action_match = re.search(r"Action:\s*(.*)",part[-1])
        action = action_match.group(1).strip()
        

        return think,action

    def _parse_action(self, action_text):
        tool_name = action_text.split("[")[0]

        tool_action = action_text.split("[")[1].replace("]", "")
        return tool_name,tool_action

        

In [13]:
from llm_client import HelloAgentsLLM
from tool_executor import executor

llm_Client = HelloAgentsLLM()
tool_executor = executor

agent = ReActAgent(llm_Client, tool_executor)

response = agent.run("The current president of the US?")

print("\n\n===== PARSE START =====")

think, action = agent._parse_output(response)

print("think:")
print(think)

print("action:")
print(action)

tool_name, tool_action = agent._parse_action(action)

print("\ntool_name:")
print(tool_name)

print("\ntool_action:")
print(tool_action)

print("===== PARSE END =====")


MODEL: openrouter/free
BASE URL: https://openrouter.ai/api/v1
KEY PREFIX: sk-or-v1
1. run() starts
2. tools: -add: add 2 numbers
-search: A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.
3. prompt:

Please note that you are an intelligent assistant capable of calling external tools.

Available tools are as follows:
-add: add 2 numbers
-search: A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.

Please respond strictly in the following format:

Thought: Your thinking process, used to analyze problems, decompose tasks, and plan the next action.
Action: The action you decide to take, must be in one of the following formats:
- {tool_name}[{tool_input}]`: Call an available tool.
- `Finish[final answer]`: When you believe you have obtained the final answer.
- When you have collected enough informati